In [1]:
import types
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from typing import Callable


In [3]:
from goatools.obo_parser import GODag
from goatools.associations import read_gaf
from goatools.semantic import TermCounts, resnik_sim, get_info_content
from itertools import product

# Source: https://notebook.community/lileiting/goatools/notebooks/semantic_similarity

enrichment_base_path = "../files/enrichment"

print("Caricamento GODag...")
godag = GODag(f"{enrichment_base_path}/go-basic.obo")
print("Caricamento GO Annotations...")
annotations = read_gaf(f"{enrichment_base_path}/goa_human.gaf") # http://current.geneontology.org/annotations/index.html
print("Applicando annotazioni al dag...")
termcounts = TermCounts(godag, annotations)
print("ALL DONE")


Caricamento GODag...
../files/enrichment/go-basic.obo: fmt(1.2) rel(2024-10-27) 44,017 Terms
Caricamento GO Annotations...
HMS:0:00:11.527675 782,823 annotations READ: ../files/enrichment/goa_human.gaf 
36340 IDs in loaded association branch, BP
Applicando annotazioni al dag...
ALL DONE


In [ ]:
def jaccard_similarity(row1: list[int], row2: list[int]) -> float:
    """
    Computa la metrica di Jaccard:
    J(A, B) = |A ∩ B| / |A ∪ B|

    :param: row1 - vettore sparso di interi
    :param: row2 - vettore sparso di interi
    :return: Jaccard similarity
    """
    intersection = np.sum(np.logical_and(row1, row2))
    union = np.sum(np.logical_or(row1, row2))
    return intersection / union if union != 0 else 0

In [ ]:
def go_heatmap_association(network_file:str, go_field:str, title:str, metric:types.FunctionType) -> None:
    network = pd.read_csv(network_file)
    
    # Togli i record che non hanno una GO
    filtered_network = network[network[go_field].notna() & (network[go_field] != '')]
    
    # Definizione dati da incrociare
    protein_list = filtered_network['Entry Name'].str[:-6].to_list()
    go_bio = filtered_network[go_field].str.findall(r'GO:\d+')
    go_list = go_bio.to_list()
    
    # Creazione matrice sparsa
    df_association = pd.DataFrame(go_list, index=protein_list).stack().reset_index()
    df_association.columns = ['Proteins', 'Index', 'GO']
    association_matrix = pd.crosstab(df_association['Proteins'], df_association['GO'])
    
    # association_matrix.to_csv('test.csv') # Visualizzazione
    
    proteins = association_matrix.index
    n = len(proteins)
    jaccard_matrix = pd.DataFrame(np.zeros((n, n)), index=proteins, columns=proteins)
    
    for i in range(n):
        for j in range(n):
            jaccard_matrix.iloc[i, j] = metric(
                association_matrix.iloc[i].values, 
                association_matrix.iloc[j].values
            )
    
    plt.figure(figsize=(20, 20))
    sns.heatmap(jaccard_matrix, cmap="viridis")
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    plt.title(title)
    plt.show()

In [ ]:
# Ottieni una mappa del tipo {Proteina1: [Go1, Go2, Go3, ...], Proteina2: [Go4, Go5, Go6, ...], ...}
# Per ogni coppia di proteine estrai le annotazioni [Go1, Go2, Go3, ...], [Go4, Go5, Go6, ...]
# Confronta a coppie con la metrica di resnik, prendi il massimo punteggio ottenuto e assegnalo alla comparazione Proteina1-Proteina2

def map_protein_go(network_file:str) -> dict[str, list[str]]:
    """
    Ritorna un dizionario dove le chiavi sono proteine e i valori sono liste di Gene Ontology IDs.
    """
    go_id_field = 'Gene Ontology IDs'
    protein_field = 'Entry Name'
    network = pd.read_csv(network_file)

    # Togli i record che non hanno una GO
    filtered_network = network[network[go_id_field].notna() & (network[go_id_field] != '')]

    # Iterate through the DataFrame rows
    protein_go_dict = {}
    for _, row in filtered_network.iterrows():
        protein = row[protein_field][:-6]
        go_ids = row[go_id_field]

        # Split the GO IDs by semicolon
        go_list = [go.strip() for go in go_ids.split(";") if go.strip()]

        # Add to the dictionary
        protein_go_dict[protein] = go_list

    return protein_go_dict

In [ ]:
# PROBLEMA_1: la metrica di resnik può restituire None...perché?
#    RISPOSTA: perché non hanno MICA, imposto a zero la sim
# PROBLEMA_2: la metrica di resnik restituisce 0 su proteine uguali????
#    RISPOSTA: perché l'IC può essere 0
# PROBLEMA_3: keyError su alcune GOs
#    RISPOSTA: alcune GOs sono obsolete, posso cercare su go-basic.obo e se hanno un sostituto cambio TODO
def resnik_bma(protein1_terms:list[str], protein2_terms:list[str], godag=godag, termcounts=termcounts) -> float:
    """
    Calcola la Best Match Average (BMA).
    Partecipano alla media solo i migliori match tra le proteine di una lista prese singolarmente e quelle dell'altra prese nell'interezza.
    """
    best_matches_1 = [
        max([resnik_sim(t1, t2, godag, termcounts) if resnik_sim(t1, t2, godag, termcounts) else 0 for t2 in protein2_terms ])
        for t1 in protein1_terms
    ]

    # Miglior match per ogni proteina nella seconda lista verso quelle della prima
    best_matches_2 = [
        max(resnik_sim(t2, t1, godag, termcounts) if resnik_sim(t2, t1, godag, termcounts) else 0 for t1 in protein1_terms)
        for t2 in protein2_terms
    ]

    # Media tra le due liste precedenti
    bma_similarity = (
        sum(best_matches_1) / len(protein1_terms) if protein1_terms else 0
        + sum(best_matches_2) / len(protein2_terms) if protein2_terms else 0
    ) / 2

    return bma_similarity

In [ ]:
def resnik_avg(protein1_terms:list[str], protein2_terms:list[str], godag=godag, termcounts=termcounts) -> float:
    similarities = [
        resnik_sim(t1, t2, godag, termcounts)
        for t1, t2 in product(protein1_terms, protein2_terms)
    ]

    return np.mean(similarities)

In [ ]:
def resnik_max(protein1_terms:list[str], protein2_terms:list[str], godag=godag, termcounts=termcounts) -> float:
    similarities = [
        resnik_sim(t1, t2, godag, termcounts)
        for t1, t2 in product(protein1_terms, protein2_terms)
    ]

    if similarities:
        return max(similarities)
    else:
        return 0.0


In [ ]:
def protein_similarity_heatmap(protein_go_map:dict, metric: Callable, title:str) -> None:
    n = len(protein_go_map)
    heatmap_data = np.zeros((n, n))
    with tqdm(total=n*(n+1)//2, desc="Calculating similarities") as pbar:
        for i, (p1, g1) in enumerate(protein_go_map.items()):
            for j, (p2, g2) in enumerate(protein_go_map.items()):
                if i <= j:
                    bma = metric(g1, g2)
                    pbar.update(1)
                    heatmap_data[i, j] = bma
                    heatmap_data[j, i] = bma
    
    plt.figure(figsize=(8, 6))
    plt.imshow(heatmap_data, cmap="viridis", interpolation="nearest")
    plt.colorbar(label="Similarity Metric")
    plt.xticks(range(n), protein_go_map.keys(), rotation=45)
    plt.yticks(range(n), protein_go_map.keys())
    plt.title(title)
    plt.tight_layout()
    plt.show()

def protein_similarity_heatmap_scaled(protein_go_map: dict, metric: Callable, title: str) -> None:
    """
    Genera una heatmap di valori di similarità scalati in modo da avere che la similarità tra due elementi uguali sia 1
    """
    proteins = list(protein_go_map.keys())
    n = len(protein_go_map)
    heatmap_data = np.zeros((n, n))
    normalization_factors = np.zeros(n)  # Store max similarity values for normalization
    
    # Calculate similarities and normalization factors
    with tqdm(total=n * (n + 1) // 2, desc="Calculating similarities") as pbar:
        for i, (p1, g1) in enumerate(protein_go_map.items()):
            max_self_similarity = metric(g1, g1)  # Self-similarity
            normalization_factors[i] = max_self_similarity if max_self_similarity > 0 else 1
            for j, (p2, g2) in enumerate(protein_go_map.items()):
                if i <= j:
                    sim = metric(g1, g2)
                    heatmap_data[i, j] = sim
                    heatmap_data[j, i] = sim
                    pbar.update(1)
    
    # Normalize the heatmap by the diagonal values
    for i in range(n):
        for j in range(n):
            heatmap_data[i, j] /= max(normalization_factors[i], normalization_factors[j])
    
    # Plot the heatmap
    plt.figure(figsize=(8, 6))
    plt.imshow(heatmap_data, cmap="viridis", interpolation="nearest")
    plt.colorbar(label="Normalized Similarity Metric")
    plt.xticks(range(n), proteins, rotation=45, ha="right")
    plt.yticks(range(n), proteins)
    plt.title(title)
    plt.tight_layout()
    plt.show()